In [ ]:
%%writefile /kaggle/working/run_lexical.py
"""
Subprocess Dual-Output Lexical Engine
====================================
Subprocess memory isolation running 78,482-row 4-View LinearSVC (Output 1)
and 5-View BPE + SGD + Style Twin + Specialist Complementary Ensemble (Output 2).
"""

import os
import gc
import re
import sys
import time
import subprocess
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

LABELS = [
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3"
]
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}

FAMILIES = {
    "gpt": [0, 1, 2, 3, 4],
    "gemini": [5, 6, 7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
}

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - c)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-15)

def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable log-softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    return x - c - np.log(np.sum(np.exp(x - c), axis=axis, keepdims=True) + 1e-15)

STOP_WORDS = set([
    "a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "aren't",
    "as", "at", "be", "because", "been", "before", "being", "below", "between", "both", "but", "by",
    "can't", "cannot", "could", "couldn't", "did", "didn't", "do", "does", "doesn't", "doing", "don't",
    "down", "during", "each", "few", "for", "from", "further", "had", "hadn't", "has", "hasn't", "have",
    "haven't", "having", "he", "he'd", "he'll", "he's", "her", "here", "here's", "hers", "herself",
    "him", "himself", "his", "how", "how's", "i", "i'd", "i'll", "i'm", "i've", "if", "in", "into",
    "is", "isn't", "it", "it's", "its", "itself", "let's", "me", "more", "most", "mustn't", "my",
    "myself", "no", "nor", "not", "of", "off", "on", "once", "only", "or", "other", "ought", "our",
    "ours", "ourselves", "out", "over", "own", "same", "shan't", "she", "she'd", "she'll", "she's",
    "should", "shouldn't", "so", "some", "such", "than", "that", "that's", "the", "their", "theirs",
    "them", "themselves", "then", "there", "there's", "these", "they", "they'd", "they'll", "they're",
    "they've", "this", "those", "through", "to", "too", "under", "until", "up", "very", "was", "wasn't",
    "we", "we'd", "we'll", "we're", "we've", "were", "weren't", "what", "what's", "when", "when's",
    "where", "where's", "which", "while", "who", "who's", "whom", "why", "why's", "with", "won't",
    "would", "wouldn't", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself"
])

def shape_text(t: str) -> str:
    """Replaces alphabetic character blocks with X (uppercase) and x (lowercase)."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m.group(0)[0].isupper() else "x", str(t))

def function_text(t: str) -> str:
    """Masks non-stopwords and non-punctuation to CONTENT, preserving function and punctuation syntax."""
    tokens = re.findall(r"(?u)\b\w+\b|[^\w\s]", str(t))
    return " ".join([w.lower() if (w.lower() in STOP_WORDS or not w.isalnum()) else "CONTENT" for w in tokens])

def run_lexical_engine(train_path: Path, test_path: Path, pool_file: Optional[Path],
                       qwen_tok_dir: str, working_dir: Path) -> Dict[str, np.ndarray]:
    """
    Runs the complete dual-output lexical engine in-process or writes standalone script.
    Saves and returns:
    - log_p_svc_main.npy (Output 1 for LOGIT_MAIN)
    - p_lex_comp.npy (Output 2 for COMP_LIST)
    - D_tfidf.npy
    - groups_085.npy
    """
    df_tr = pd.read_csv(train_path)
    df_te = pd.read_csv(test_path)

    test_texts = df_te['text'].fillna('').astype(str).tolist()
    fit_texts = df_tr['text'].fillna('').astype(str).tolist()
    fit_y = np.array([LAB2I[m] for m in df_tr['model']])
    len_tr_raw = len(df_tr)

    # 1. Compute stopword TF-IDF distance matrix and topic clusters (t=0.85) on test
    Xt_tfidf = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=100000, dtype=np.float32).fit_transform(test_texts)
    if len(test_texts) >= 2:
        D = np.clip(1.0 - (Xt_tfidf @ Xt_tfidf.T).toarray(), 0.0, None)
        np.fill_diagonal(D, 0.0)
        cond_d = squareform(D, checks=False)
        groups_085 = fcluster(linkage(cond_d, method='average'), 0.85, criterion='distance')
    else:
        D = np.zeros((1, 1), dtype=np.float32)
        groups_085 = np.ones(1, dtype=int)

    sz_085 = np.bincount(groups_085)[groups_085]
    twin_rows_085 = (sz_085 >= 2) & (sz_085 <= 18)

    # 2. Load pool_v4.parquet if present
    if pool_file and os.path.exists(pool_file):
        df_pool = pd.read_parquet(pool_file, columns=['text', 'model'])
        df_pool = df_pool[df_pool['model'].isin(LABELS)].dropna(subset=['text', 'model']).copy()
        fit_texts += df_pool['text'].fillna('').astype(str).tolist()
        fit_y = np.r_[fit_y, df_pool['model'].map(LAB2I).values]
        del df_pool
        gc.collect()

    # 3. Canonical 4 Views (from NTL two-stage LoRA part two)
    VIEWS_4V = [
        ("char", 3, 5, 250000, 1.0),
        ("word", 1, 3, 180000, 0.7),
        ("shape", 2, 6, 60000, 0.5),
        ("function", 1, 3, 40000, 0.5),
    ]

    fit_sh = [shape_text(t) for t in fit_texts]
    test_sh = [shape_text(t) for t in test_texts]
    fit_fn = [function_text(t) for t in fit_texts]
    test_fn = [function_text(t) for t in test_texts]

    tr_mats_4v = []
    te_mats_4v = []
    for mode, mn, mx, mf, sc in VIEWS_4V:
        if mode == "shape":
            v = TfidfVectorizer(analyzer="char", ngram_range=(mn, mx), max_features=mf, sublinear_tf=True, dtype=np.float32)
            tr_m = v.fit_transform(fit_sh) * sc
            te_m = v.transform(test_sh) * sc
        elif mode == "function":
            v = TfidfVectorizer(analyzer="word", ngram_range=(mn, mx), max_features=mf, sublinear_tf=True, dtype=np.float32)
            tr_m = v.fit_transform(fit_fn) * sc
            te_m = v.transform(test_fn) * sc
        elif mode == "word":
            v = TfidfVectorizer(analyzer="word", ngram_range=(mn, mx), max_features=mf,
                                token_pattern=r"(?u)\b\w+\b|[^\w\s]", lowercase=False,
                                sublinear_tf=True, dtype=np.float32)
            tr_m = v.fit_transform(fit_texts) * sc
            te_m = v.transform(test_texts) * sc
        else:
            v = TfidfVectorizer(analyzer=mode, ngram_range=(mn, mx), max_features=mf, sublinear_tf=True, dtype=np.float32)
            tr_m = v.fit_transform(fit_texts) * sc
            te_m = v.transform(test_texts) * sc
        tr_mats_4v.append(tr_m)
        te_mats_4v.append(te_m)

    X_tr_4v = sparse.hstack(tr_mats_4v, format="csr")
    X_te_4v = sparse.hstack(te_mats_4v, format="csr")

    # OUTPUT 1: Pure 4-View LinearSVC (C=0.5, uniform 1.0 sample weights)
    svc_main = LinearSVC(C=0.5, random_state=42, max_iter=3000, tol=1e-3)
    svc_main.fit(X_tr_4v, fit_y)
    logits_svc_main = svc_main.decision_function(X_te_4v)
    log_p_svc_main = log_softmax(logits_svc_main)

    # OUTPUT 2: 5-View BPE + SGD + Style Twin + Specialist Complementary Ensemble
    tok = None
    if qwen_tok_dir and os.path.exists(qwen_tok_dir):
        try:
            from transformers import AutoTokenizer
            tok = AutoTokenizer.from_pretrained(qwen_tok_dir, use_fast=True)
        except Exception:
            tok = None

    if tok is not None:
        def bpe_tokenize(txt: str) -> List[str]:
            return [str(tok_id) for tok_id in tok.encode(str(txt), truncation=True, max_length=512)]
        v_bpe = TfidfVectorizer(tokenizer=bpe_tokenize, lowercase=False, token_pattern=None,
                                ngram_range=(1, 3), max_features=75000, sublinear_tf=True, dtype=np.float32)
        tr_bpe = v_bpe.fit_transform(fit_texts) * 0.8
        te_bpe = v_bpe.transform(test_texts) * 0.8
        X_tr_5v = sparse.hstack([X_tr_4v, tr_bpe], format="csr")
        X_te_5v = sparse.hstack([X_te_4v, te_bpe], format="csr")
    else:
        v_cw = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=75000, sublinear_tf=True, dtype=np.float32)
        tr_cw = v_cw.fit_transform(fit_texts) * 0.8
        te_cw = v_cw.transform(test_texts) * 0.8
        X_tr_5v = sparse.hstack([X_tr_4v, tr_cw], format="csr")
        X_te_5v = sparse.hstack([X_te_4v, te_cw], format="csr")

    svc_comp = LinearSVC(C=0.35, random_state=42, max_iter=3000, tol=1e-3)
    svc_comp.fit(X_tr_5v, fit_y)
    P_svc_comp = softmax(svc_comp.decision_function(X_te_5v), axis=1)

    sgd_comp = SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=1500, random_state=42)
    sgd_comp.fit(X_tr_5v, fit_y)
    P_sgd_comp = sgd_comp.predict_proba(X_te_5v)

    # Style Twin on groups_085
    P_twin_comp = P_svc_comp.copy()
    if twin_rows_085.any() and len_tr_raw >= 18:
        X_train_only = X_tr_5v[:len_tr_raw]
        u_tr, inv_tr = np.unique(df_tr['prompt_id'].values, return_inverse=True)
        counts_tr = np.bincount(inv_tr)
        vals_tr = (1.0 / counts_tr[inv_tr]).astype(np.float32)
        G = sparse.csr_matrix((vals_tr, (np.arange(len(inv_tr)), inv_tr)), shape=(len(inv_tr), len(u_tr)), dtype=np.float32)
        X_train_c = X_train_only - G @ (G.T @ X_train_only)

        svc_twin = LinearSVC(C=0.30, random_state=42, max_iter=2500)
        svc_twin.fit(X_train_c, fit_y[:len_tr_raw])
        del X_train_only, G, X_train_c
        gc.collect()

        X_te_twin = X_te_5v[twin_rows_085]
        u_t, inv_t = np.unique(groups_085[twin_rows_085], return_inverse=True)
        counts_t = np.bincount(inv_t)
        vals_t = (1.0 / counts_t[inv_t]).astype(np.float32)
        G_t = sparse.csr_matrix((vals_t, (np.arange(len(inv_t)), inv_t)), shape=(len(inv_t), len(u_t)), dtype=np.float32)
        X_te_twin_c = X_te_twin - G_t @ (G_t.T @ X_te_twin)
        P_twin_comp[twin_rows_085] = softmax(svc_twin.decision_function(X_te_twin_c), axis=1)
        del svc_twin, X_te_twin, G_t, X_te_twin_c
        gc.collect()

    P_lex_comp = 0.50 * P_svc_comp + 0.35 * P_sgd_comp + 0.15 * P_twin_comp
    log_p_lex_comp = np.log(np.clip(P_lex_comp, 1e-9, 1.0))

    # 4 Within-Family Specialist Refiners
    beta = 0.25
    for fam_name, fam_cols in FAMILIES.items():
        fam_mask = np.isin(fit_y, fam_cols)
        if fam_mask.sum() > 50:
            sub_y = fit_y[fam_mask]
            sub_lab2i = {c: i for i, c in enumerate(fam_cols)}
            sub_y_mapped = np.array([sub_lab2i[y] for y in sub_y])

            spec_svc = LinearSVC(C=0.40, random_state=42, max_iter=2000, tol=1e-3, class_weight="balanced")
            spec_svc.fit(X_tr_5v[fam_mask], sub_y_mapped)
            df_spec = spec_svc.decision_function(X_te_5v)
            logits_spec = np.column_stack([-df_spec, df_spec]) if df_spec.ndim == 1 else df_spec
            P_spec = softmax(logits_spec, axis=1)

            P_prior = P_lex_comp[:, fam_cols]
            P_prior_norm = P_prior / (P_prior.sum(axis=1, keepdims=True) + 1e-12)
            P_refined = (1.0 - beta) * P_prior_norm + beta * P_spec

            log_ratio = np.log(np.clip(P_refined, 1e-9, 1.0)) - np.log(np.clip(P_prior_norm, 1e-9, 1.0))
            log_p_lex_comp[:, fam_cols] += log_ratio

    P_lex_comp_final = softmax(log_p_lex_comp, axis=1)

    # Save to disk
    os.makedirs(working_dir, exist_ok=True)
    np.save(working_dir / 'log_p_svc_main.npy', log_p_svc_main)
    np.save(working_dir / 'p_lex_comp.npy', P_lex_comp_final)
    np.save(working_dir / 'D_tfidf.npy', D)
    np.save(working_dir / 'groups_085.npy', groups_085)

    return {
        "log_p_svc_main": log_p_svc_main,
        "p_lex_comp": P_lex_comp_final,
        "D_tfidf": D,
        "groups_085": groups_085
    }

if __name__ == "__main__":
    working_dir = Path("/kaggle/working")
    if not working_dir.exists():
        working_dir = Path("local_data")
    working_dir.mkdir(parents=True, exist_ok=True)

    input_dir = Path("/kaggle/input")
    if not input_dir.exists():
        input_dir = Path(".")

    candidates = [
        input_dir / 'competitions' / 'name-that-llm',
        input_dir / 'name-that-llm',
        Path('local_data'),
        Path('.')
    ]
    train_path = None
    test_path = None
    for c in candidates:
        if (c / 'train.csv').exists() and train_path is None:
            train_path = c / 'train.csv'
        if (c / 'test.csv').exists() and test_path is None:
            test_path = c / 'test.csv'
    if train_path is None:
        train_path = Path('local_data') / 'train.csv'
    if test_path is None:
        test_path = Path('local_data') / 'test.csv'

    pool_paths = list(input_dir.rglob('pool_v4.parquet'))
    pool_file = pool_paths[0] if pool_paths else None

    tok_paths = list(input_dir.rglob("vocab.json"))
    qwen_tok_dir = str(tok_paths[0].parent) if tok_paths else ""

    print(f"[Module 1 CLI] Running lexical engine: train={train_path}, test={test_path}")
    run_lexical_engine(train_path, test_path, pool_file, qwen_tok_dir, working_dir)
    print("[Module 1 CLI] Lexical engine finished successfully.")


In [ ]:
# === MODULE: src/namethatllm/config.py ===
ENSEMBLE_CONFIG = {'version': '1.0.0', 'name': 'production_ensemble', 'description': 'Production Two-Tier Stack (85% Main Part-Two Baseline + 15% Prior-Scaled Complementary Stack + Unsupervised Spectral Bank)', 'labels': ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'], 'families': {'gpt': [0, 1, 2, 3, 4], 'gemini': [5, 6, 7, 8], 'claude': [9, 10, 11, 12, 13], 'glm': [15, 16]}, 'main_stack': {'linear_svc_weight': 1.0, 'members': {'lora_qwen3_14b': {'plain_weight': 0.4, 'twin_weight': 0.1, 'twin_clustering': 'groups_085'}, 'lora_qwen25_14b': {'plain_weight': 0.2, 'twin_weight': 0.1, 'twin_clustering': 'groups_085'}, 'deberta_v3_large': {'plain_weight': 0.25, 'twin_weight': 0.0}}}, 'complementary_stack': {'stream_weights': {'lora_and_twins': 0.35, 'lexical': 0.2, 'dp_posteriors': 0.3, 'spectral_bank': 0.15}, 'lora_members': {'lora_qwen3_14b': {'plain_weight': 0.3, 'twin_weight': 0.1}, 'lora14b_v1': {'plain_weight': 0.14, 'twin_weight': 0.05}, 'lora_qwen25_14b': {'plain_weight': 0.2, 'twin_weight': 0.08}, 'instyle': {'plain_weight': 0.22, 'twin_weight': 0.08}, 'ntl_claude_head': {'plain_weight': 0.14, 'twin_weight': 0.05}, 'deberta_v3_large': {'plain_weight': 0.18, 'twin_weight': 0.06}}, 'twin_resolutions': {'t85': 0.5, 't72': 0.3, 't60': 0.2}, 'dp_weights': {'q3_14b': 0.38, 'q25_14b': 0.24, 'deberta': 0.2, 'instyle': 0.1, 'part1_14b': 0.08}, 'se_floors': {'qwen3_14b_p2': 0.1, 'qwen25_14b_p2': 0.1, 'deberta_p2': 0.1, 'qwen3_14b_p1': 0.35, 'instyle': 0.35, 'qwen3_8b_claude': 0.35}, 'prior_scales': {'qwen3_14b_p2': 1.0, 'qwen25_14b_p2': 0.85, 'deberta_p2': 0.8, 'qwen3_14b_p1': 0.5, 'instyle': 0.45, 'qwen3_8b_claude': 0.3}, 'spectral_bank': {'plain_C': 0.04, 'center_C': 0.03, 'center_blend': 0.35}}, 'fusion': {'main_ratio': 0.85, 'comp_ratio': 0.15}, 'postprocess': {'claude_specialist_weight': 0.15, 'curly_quote_multiplier': 0.36787944117144233, 'sinkhorn_iterations': 300, 'max_cluster_size': 14}}

"""
Canonical Configuration, Constants & Path Resolution
====================================================
Master canonical label ordering, provider families, simplex helpers,
and Kaggle filesystem asset discovery.
"""

import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
import numpy as np

# Canonical 18-Class Labels (Strict order from NTL two-stage LoRA part two)
LABELS = [
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3"
]
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}

FAMILIES = {
    "gpt": [0, 1, 2, 3, 4],
    "gemini": [5, 6, 7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
}

PRIOR_SCALES = {
    "qwen3_14b_p2": 1.00,
    "qwen25_14b_p2": 0.85,
    "deberta_p2": 0.80,
    "qwen3_14b_p1": 0.50,
    "instyle": 0.45,
    "qwen3_8b_claude": 0.30
}

def align_cols(arr: np.ndarray, src_labels: list) -> np.ndarray:
    """Aligns columns of array to canonical 18-class order."""
    src_list = [x.decode('utf-8') if isinstance(x, (bytes, np.bytes_)) else str(x) for x in src_labels]
    idx = [src_list.index(l) for l in LABELS]
    if arr.ndim == 1:
        return arr[idx]
    elif arr.ndim == 2:
        if arr.shape[0] == len(src_list) and arr.shape[1] != len(src_list):
            return arr[idx, :]
        elif arr.shape[1] == len(src_list):
            return arr[:, idx]
    return arr[..., idx]

def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable log-softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    return x - c - np.log(np.sum(np.exp(x - c), axis=axis, keepdims=True) + 1e-15)

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - c)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-15)

def resolve_dataset_paths(input_dir: Optional[Path] = None) -> Tuple[Path, Path, Optional[Path]]:
    """Resolves train.csv, test.csv, and pool_v4.parquet across local and Kaggle environments."""
    if input_dir is None:
        input_dir = Path('/kaggle/input')
        if not input_dir.exists():
            input_dir = Path('.')

    candidates = [
        input_dir / 'competitions' / 'name-that-llm',
        input_dir / 'name-that-llm',
        Path('local_data'),
        Path('.')
    ]
    train_path = None
    test_path = None
    for c in candidates:
        if (c / 'train.csv').exists() and train_path is None:
            train_path = c / 'train.csv'
        if (c / 'test.csv').exists() and test_path is None:
            test_path = c / 'test.csv'

    if train_path is None:
        train_path = Path('local_data') / 'train.csv'
    if test_path is None:
        test_path = Path('local_data') / 'test.csv'

    pool_paths = list(input_dir.rglob('pool_v4.parquet'))
    pool_file = pool_paths[0] if pool_paths else None

    return train_path, test_path, pool_file

def find_base_model(input_dir: Path, model_family: str, hidden_sz: int, num_layers: int) -> Optional[Path]:
    """Discovers base model directories from Hugging Face config.json inspection."""
    for cfg_p in input_dir.rglob('config.json'):
        try:
            cfg = json.load(open(cfg_p))
            m_type = str(cfg.get('model_type', '')).lower()
            h_dim = cfg.get('hidden_size', 0)
            n_lay = cfg.get('num_hidden_layers', 0)
            if model_family.lower() in m_type and h_dim == hidden_sz and n_lay == num_layers:
                return cfg_p.parent
        except Exception:
            continue
    return None

def find_checkpoints(input_dir: Path) -> Dict[str, Path]:
    """Finds all fine-tuned LoRA and head checkpoints."""
    ckpts = {}
    for p in input_dir.rglob('heads.safetensors'):
        p_str = str(p).lower().replace('\\', '/')
        if 'lora_qwen3_14b' in p_str or 'qwen3_14b_part2' in p_str or ('14b' in p_str and 'part2' in p_str and 'qwen3' in p_str):
            ckpts['qwen3_14b_p2'] = p.parent
        elif 'lora_qwen25_14b' in p_str or 'qwen25_14b_part2' in p_str or ('14b' in p_str and 'part2' in p_str and 'qwen2' in p_str):
            ckpts['qwen25_14b_p2'] = p.parent
        elif 'lora14b_v1' in p_str:
            ckpts['qwen3_14b_p1'] = p.parent
        elif 'lora8b_v1' in p_str:
            ckpts['qwen3_8b_p1'] = p.parent
        elif 'instyle' in p_str:
            ckpts['instyle'] = p.parent

    for p in input_dir.rglob('claude_head.safetensors'):
        ckpts['qwen3_8b_claude'] = p.parent
        break
    if 'qwen3_8b_claude' not in ckpts:
        for p in input_dir.rglob('readouts.npz'):
            if 'claude' in str(p).lower():
                ckpts['qwen3_8b_claude'] = p.parent
                break

    for p in input_dir.rglob('head.safetensors'):
        if 'deberta' in str(p).lower():
            ckpts['deberta_p2'] = p.parent
            break

    if 'instyle' not in ckpts:
        for p in input_dir.rglob('*'):
            p_s = str(p).lower().replace('\\', '/')
            if 'instyle' in p_s and ((p / 'adapter_model.safetensors').exists() or (p / 'adapter' / 'adapter_model.safetensors').exists() or (p / 'heads.safetensors').exists()):
                ckpts['instyle'] = p
                break
    return ckpts


In [ ]:
# === MODULE: src/namethatllm/wpca_gaussian.py ===
"""
Full-Rank wPCA & ExactAuthorPosterior (2^18 Bitmask DP)
======================================================
Two-way variance decomposition, Ledoit-Wolf shrinkage floors,
crash-proof cluster splitting (max_size <= 14), and exact forward-backward
bitmask DP author posteriors under the unique-authorship constraint.
"""

from functools import lru_cache
from typing import Tuple, List, Dict, Optional, Any
import numpy as np
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.special import logsumexp
from sklearn.covariance import ledoit_wolf_shrinkage


def _two_way(Z: np.ndarray, y: np.ndarray, pinv: np.ndarray, G: int, K_classes: int = 18, iters: int = 4) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Iterative two-way ANOVA decomposition separating author means and prompt random effects."""
    N, d = Z.shape
    mu = np.zeros((K_classes, d), dtype=np.float64)
    P = np.zeros((G, d), dtype=np.float64)
    ng = np.bincount(pinv, minlength=G).clip(min=1)
    for _ in range(iters):
        R = Z - P[pinv]
        for k in range(K_classes):
            idx = (y == k)
            if idx.any():
                mu[k] = R[idx].mean(axis=0)
        R = Z - mu[y]
        P[:] = 0.0
        np.add.at(P, pinv, R)
        P /= ng[:, None]
    return mu, P, ng

def _shrunk(S: np.ndarray, alpha: float) -> np.ndarray:
    """Shrinks covariance matrix S towards spherical target."""
    d = S.shape[0]
    tr = np.trace(S) / d
    return (1.0 - alpha) * S + alpha * tr * np.eye(d)

def fit_shared_gaussian_lowram(X_tr_f32: np.ndarray, y: np.ndarray, prompts: np.ndarray,
                               d_target: int = 256, se_floor: float = 0.10) -> dict:
    """
    Fits shared Gaussian whitener with within-prompt centering and Ledoit-Wolf shrinkage floor.
    Returns unsupervised top-64 within-prompt principal components V_top64 (zero label leakage!).
    """
    N, D = X_tr_f32.shape
    pu, pinv = np.unique(prompts, return_inverse=True)
    G = len(pu)

    mx = X_tr_f32.mean(axis=0, dtype=np.float64).astype(np.float32)
    sx = X_tr_f32.std(axis=0, dtype=np.float64).astype(np.float32)
    sx = np.where(sx > 1e-6, sx, 1.0).astype(np.float32)

    # Standardize and subtract prompt means in-place (Within-Prompt Whitened PCA!)
    R = (X_tr_f32 - mx) / sx
    Pm = np.zeros((G, D), dtype=np.float32)
    np.add.at(Pm, pinv, R)
    Pm /= np.bincount(pinv, minlength=G)[:, None].clip(min=1).astype(np.float32)
    R -= Pm[pinv]
    del Pm

    d = int(min(d_target, max(N - G - 1, 1), D))
    Gram = (R @ R.T).astype(np.float64)
    lam_r, U = np.linalg.eigh(Gram)
    idx_top = np.argsort(lam_r)[::-1][:d]
    lam_top = np.clip(lam_r[idx_top], 1e-10, None)
    U_f32 = U[:, idx_top].astype(np.float32)
    V_f32 = (R.T @ U_f32) / np.sqrt(lam_top).astype(np.float32)[None, :]
    del R, Gram, U, U_f32

    # Unsupervised within-prompt top-64 principal components (zero label leakage)
    V_top64 = V_f32[:, :min(64, d)].copy()

    Z = (((X_tr_f32 - mx) / sx) @ V_f32).astype(np.float64)

    mu, P, ng = _two_way(Z, y, pinv, G, K_classes=K, iters=4)
    E = Z - mu[y] - P[pinv]
    dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = (E.T @ E) / dof
    ae = max(float(ledoit_wolf_shrinkage(E, assume_centered=True)), se_floor)
    Se = _shrunk(Se_raw, ae)

    Pc = P - P.mean(0)
    Sp_raw = (Pc.T @ Pc) / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = float(ledoit_wolf_shrinkage(Pc, assume_centered=True))
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se))
    Sp = (1.0 - ap) * Sp_raw + ap * (tr / d) * np.eye(d)

    L = np.linalg.cholesky(Se)
    Li = np.linalg.inv(L)
    M = 0.5 * ((Li @ Sp @ Li.T) + (Li @ Sp @ Li.T).T)
    lam, Q = np.linalg.eigh(M)
    lam = np.clip(lam, 1e-4, None)
    A = Q.T @ Li
    W_f32 = (V_f32 @ A.T.astype(np.float32)).astype(np.float32)
    m_whitened = mu @ A.T

    return {
        "mx": mx, "sx": sx, "W": W_f32, "m": m_whitened, "lam": lam,
        "V_top64": V_top64, "T1": 2.40, "Tg": 2.35, "ae": ae
    }

def transform_lowram(X_f32: np.ndarray, prm: dict, batch_rows: int = 1000) -> np.ndarray:
    """Projects representations into whitened space."""
    N = len(X_f32)
    d = prm["W"].shape[1]
    U_out = np.empty((N, d), dtype=np.float64)
    mx, sx, W = prm["mx"], prm["sx"], prm["W"]
    for s in range(0, N, batch_rows):
        chunk = ((X_f32[s : s + batch_rows] - mx) / sx) @ W
        U_out[s : s + batch_rows] = chunk.astype(np.float64)
    return U_out

def split_oversized_clusters_dist(D: np.ndarray, groups: np.ndarray, max_cluster_size: int = 14) -> np.ndarray:
    """Recursively splits clusters with size > max_cluster_size to prevent bitmask DP overflow."""
    if len(groups) < 2:
        return groups.copy()
    new_groups = np.zeros(len(groups), dtype=np.int64)
    next_id = 1
    for g in np.unique(groups):
        rows = np.flatnonzero(groups == g)
        if len(rows) <= max_cluster_size:
            new_groups[rows] = next_id
            next_id += 1
        else:
            n_sub = int(np.ceil(len(rows) / max_cluster_size))
            D_sub = D[np.ix_(rows, rows)].copy()
            np.fill_diagonal(D_sub, 0.0)
            D_sub = np.clip(0.5 * (D_sub + D_sub.T), 0.0, None)
            d_condensed = squareform(D_sub, checks=False)
            sub_labels = fcluster(linkage(d_condensed, method='average'), t=n_sub, criterion='maxclust')
            for sg in np.unique(sub_labels):
                s_rows = rows[sub_labels == sg]
                for start in range(0, len(s_rows), max_cluster_size):
                    new_groups[s_rows[start : start + max_cluster_size]] = next_id
                    next_id += 1
    return new_groups

@lru_cache(maxsize=4)
def _bitmask_levels(k: int = 18):
    """Precomputes bitmasks partitioned by population count (number of set bits)."""
    masks = np.arange(1 << k, dtype=np.int64)
    counts = np.fromiter((int(x).bit_count() for x in masks), dtype=np.uint8, count=len(masks))
    return tuple(masks[counts == n] for n in range(k + 1))

class ExactAuthorPosterior:
    """Exact shared-prompt Gaussian posteriors under author uniqueness via 2^18 bitmask DP."""
    def __init__(self, m: np.ndarray, lam: np.ndarray):
        self.m = np.asarray(m, dtype=np.float64)
        self.lam = np.asarray(lam, dtype=np.float64)
        self.K, self.d = self.m.shape
        self.levels = _bitmask_levels(self.K)
        self._subset_means = {}

    def single(self, u: np.ndarray, temperature: float = 2.40) -> np.ndarray:
        diff = u[None, :] - self.m
        dist = np.sum((diff * diff) / (1.0 + self.lam)[None, :], axis=1)
        log_p = -0.5 * dist / max(float(temperature), 1e-4)
        return log_p - logsumexp(log_p)

    def group(self, U: np.ndarray, temperature: float = 2.35) -> np.ndarray:
        U = np.asarray(U, dtype=np.float64)
        n = len(U)
        if n <= 1 or n > self.K:
            return np.vstack([self.single(U[i], temperature=temperature) for i in range(n)])

        if n not in self._subset_means:
            if len(self._subset_means) >= 3:
                self._subset_means.clear()
            masks = self.levels[n]
            membership = ((masks[:, None] >> np.arange(self.K)) & 1).astype(np.float32)
            self._subset_means[n] = membership @ self.m.astype(np.float32)

        terminal = self.levels[n]
        gamma = self.lam / (1.0 + n * self.lam)
        residual = U.sum(axis=0, keepdims=True) - self._subset_means[n].astype(np.float64)
        correction = 0.5 * np.einsum("nd,d,nd->n", residual, gamma, residual) / max(float(temperature), 1e-4)
        correction -= correction.max()

        # Passage-specific unary log-likelihoods A[i, author]
        A = -0.5 * np.sum((U[:, None, :] - self.m[None, :, :]) ** 2, axis=2) / max(float(temperature), 1e-4)
        A -= A.max(axis=1, keepdims=True)

        forward = np.full(1 << self.K, -np.inf, dtype=np.float64)
        forward[0] = 0.0
        for count in range(1, n + 1):
            masks = self.levels[count]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) != 0]
                forward[chosen] = np.logaddexp(forward[chosen], forward[chosen ^ bit] + A[count - 1, author])

        backward = np.full_like(forward, -np.inf)
        backward[terminal] = correction
        for count in range(n - 1, -1, -1):
            masks = self.levels[count]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) == 0]
                backward[chosen] = np.logaddexp(backward[chosen], A[count, author] + backward[chosen | bit])

        result = np.empty((n, self.K), dtype=np.float64)
        for row in range(n):
            masks = self.levels[row]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) == 0]
                result[row, author] = logsumexp(forward[chosen] + A[row, author] + backward[chosen | bit]) - backward[0]

        result -= logsumexp(result, axis=1, keepdims=True)
        return result

def scores_exact_dp(U_test: np.ndarray, groups: np.ndarray, prm: dict, D_dist: np.ndarray) -> np.ndarray:
    """Wraps ExactAuthorPosterior with oversized cluster splitting guardrail."""
    N = len(U_test)
    if N == 0:
        return np.zeros((0, K), dtype=np.float64)
    if N == 1:
        solver = ExactAuthorPosterior(prm["m"], prm["lam"])
        return solver.single(U_test[0], temperature=prm.get("T1", 2.40))[None, :]

    safe_groups = split_oversized_clusters_dist(D_dist, groups, max_cluster_size=14)
    solver = ExactAuthorPosterior(prm["m"], prm["lam"])
    T1 = prm.get("T1", 2.40)
    Tg = prm.get("Tg", 2.35)

    lp_dp = np.zeros((N, K), dtype=np.float64)
    for g in np.unique(safe_groups):
        idx_g = np.where(safe_groups == g)[0]
        if len(idx_g) == 1:
            lp_dp[idx_g] = solver.single(U_test[idx_g[0]], temperature=T1)
        else:
            lp_dp[idx_g] = solver.group(U_test[idx_g], temperature=Tg)
    return lp_dp


In [ ]:
# === MODULE: src/namethatllm/spectral_bank.py ===
"""
Leak-Free Prior-Scaled 384-Dim Joint Spectral Bank
=================================================
Constructs full-rank cross-backbone spectral representations from unsupervised
prompt-centered principal components V_top64, scaled by out-of-sample prior scales.
"""

from typing import List, Tuple, Dict, Any, Optional
import numpy as np
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression


def fit_joint_spectral_bank(
    spectral_tr_blocks: List[np.ndarray],
    spectral_te_blocks: List[np.ndarray],
    y_sub: np.ndarray,
    prompts_sub: np.ndarray,
    D_tfidf: np.ndarray,
    D_neural_sum: np.ndarray,
    D_neural_weight_total: float,
    N_test: int,
    plain_C: float = 0.04,
    center_C: float = 0.03,
    center_blend: float = 0.35,
    max_cluster_size: int = 14
) -> Tuple[np.ndarray, float]:
    """
    Fits plain and prompt-centered logistic regressions on stacked unsupervised spectral features.
    Returns:
    - P_spectral: (N_test, K) predicted probabilities
    - oof_bacc: 5-fold GroupKFold balanced accuracy
    """
    if len(spectral_tr_blocks) == 0:
        return np.ones((N_test, K), dtype=np.float64) / float(K), 0.0

    X_spec_tr = np.hstack(spectral_tr_blocks)
    X_spec_te = np.hstack(spectral_te_blocks)

    # 5-fold GroupKFold OOF Balanced Accuracy (zero label leakage)
    gkf = GroupKFold(n_splits=5)
    oof_preds = np.zeros(len(y_sub), dtype=int)
    for tr_idx, val_idx in gkf.split(X_spec_tr, y_sub, groups=prompts_sub):
        clf_fold = LogisticRegression(C=plain_C, max_iter=2000, random_state=42)
        clf_fold.fit(X_spec_tr[tr_idx], y_sub[tr_idx])
        oof_preds[val_idx] = clf_fold.predict(X_spec_tr[val_idx])
    oof_bacc = float(balanced_accuracy_score(y_sub, oof_preds))

    # Plain Logistic Regression
    clf_plain = LogisticRegression(C=plain_C, max_iter=2000, random_state=42)
    clf_plain.fit(X_spec_tr, y_sub)
    lp_spec_plain = clf_plain.predict_log_proba(X_spec_te)
    if hasattr(clf_plain, 'classes_'):
        lp_spec_plain = align_cols(lp_spec_plain, [LABELS[c] for c in clf_plain.classes_])

    # Prompt-Centered Logistic Regression
    X_spec_tr_c = X_spec_tr.copy()
    for p in np.unique(prompts_sub):
        mask = (prompts_sub == p)
        X_spec_tr_c[mask] -= X_spec_tr_c[mask].mean(axis=0, keepdims=True)
    clf_center = LogisticRegression(C=center_C, max_iter=2000, random_state=42)
    clf_center.fit(X_spec_tr_c, y_sub)

    # Prompt-centered test features
    X_spec_te_c = X_spec_te.copy()
    D_neural_final = D_neural_sum / max(1e-6, D_neural_weight_total) if D_neural_weight_total > 0 else D_tfidf
    D_hybrid_final = 0.55 * D_tfidf + 0.45 * D_neural_final
    g65_final = fcluster(linkage(squareform(D_hybrid_final, checks=False), method='average'), t=0.65, criterion='distance') if N_test >= 2 else np.ones(N_test, dtype=int)
    safe_groups = split_oversized_clusters_dist(D_hybrid_final, g65_final, max_cluster_size=max_cluster_size)

    if N_test >= 2 and len(np.unique(safe_groups)) > 0:
        for g in np.unique(safe_groups):
            mask = (safe_groups == g)
            if mask.sum() > 1:
                X_spec_te_c[mask] -= X_spec_te_c[mask].mean(axis=0, keepdims=True)
            else:
                X_spec_te_c[mask] = 0.0
    else:
        X_spec_te_c = np.zeros_like(X_spec_te)

    lp_spec_center = clf_center.predict_log_proba(X_spec_te_c)
    if hasattr(clf_center, 'classes_'):
        lp_spec_center = align_cols(lp_spec_center, [LABELS[c] for c in clf_center.classes_])

    lp_spec_joint = lp_spec_plain + center_blend * lp_spec_center
    P_spectral = softmax(lp_spec_joint, axis=1)

    return P_spectral, oof_bacc


In [ ]:
# === MODULE: src/namethatllm/neural_extractor.py ===
"""
Universal Neural Trunk Loader & Feature Extractor
=================================================
Manages multi-GPU sharding (2xT4 GPUs, max 13.5 GiB), layer truncation (keep_layers),
dynamic bfloat16 row-level retry for 14B fp16 overflow rescue, LayerNorm heads,
and bit-exact CPU fp16 snapshot/restore for shared trunks.
"""

import gc
import os
import json
import time
import shutil
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel, AutoTokenizer
from safetensors.torch import load_file


def in_place_copy(dst: torch.Tensor, src: torch.Tensor):
    """In-place weight copy."""
    dst.data.copy_(src.to(device=dst.device, dtype=dst.dtype))

def merge_lora(model: nn.Module, adapter_path: Path, sign: float = +1.0) -> bool:
    """In-place merges or unmerges (+1.0 or -1.0) a LoRA adapter directly into trunk weights."""
    weights_path = adapter_path / "adapter_model.safetensors"
    if not weights_path.exists():
        weights_path = adapter_path / "adapter" / "adapter_model.safetensors"
    if not weights_path.exists():
        return False

    sd = load_file(str(weights_path))
    cfg_p = adapter_path / "adapter_config.json"
    if not cfg_p.exists():
        cfg_p = adapter_path / "adapter" / "adapter_config.json"
    scale = 1.0
    if cfg_p.exists():
        c = json.load(open(cfg_p))
        scale = float(c.get("lora_alpha", 16)) / float(c.get("r", 16))

    state_dict = dict(model.named_modules())
    pairs = {}
    for k in sd.keys():
        if "lora_A" in k:
            b_key = k.replace("lora_A", "lora_B")
            if b_key in sd:
                m_name = k.split(".lora_A")[0].replace("base_model.model.", "")
                pairs[m_name] = (sd[k], sd[b_key])

    applied = 0
    with torch.no_grad():
        for m_name, (A, B) in pairs.items():
            if m_name in state_dict:
                mod = state_dict[m_name]
                if hasattr(mod, "weight"):
                    A_d = A.to(mod.weight.device, dtype=torch.float32)
                    B_d = B.to(mod.weight.device, dtype=torch.float32)
                    delta = (B_d @ A_d) * (scale * sign)
                    mod.weight.data.add_(delta.to(mod.weight.dtype))
                    applied += 1
    return applied > 0

@torch.no_grad()
def forward_and_pool_batch(model: nn.Module, enc: dict, layers: List[int], norms: nn.ModuleList,
                           dtype: torch.dtype = torch.float16, drop_first_token: bool = True):
    """
    Runs forward pass in `dtype` (float16 on T4).
    CRITICAL: If any target layer overflows fp16 (Inf/NaN), recomputes ONLY the overflowed
    rows in `torch.bfloat16` and pools directly in float32 before restoring `dtype`!
    """
    hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states

    B = enc["input_ids"].shape[0]
    first_norm_dev = norms[0].weight.device

    # Check if any selected layer has Inf/NaN per row
    bad_mask = torch.zeros(B, dtype=torch.bool, device="cpu")
    for l in layers:
        bad_mask |= ~torch.isfinite(hs[l]).all(dim=(1, 2)).cpu()

    n_bad = int(bad_mask.sum().item())
    m = enc["attention_mask"].unsqueeze(-1).float().to(first_norm_dev)
    if drop_first_token:
        m[:, 0, :] = 0.0
    den = m.sum(dim=1).clamp(min=1.0)

    # Pool finite rows first in float32
    pooled_layers = [torch.zeros((B, model.config.hidden_size), dtype=torch.float32, device=first_norm_dev) for _ in layers]
    good_rows = torch.where(~bad_mask)[0].to(first_norm_dev)
    if len(good_rows) > 0:
        for idx_l, l in enumerate(layers):
            h_good = hs[l][good_rows.to(hs[l].device)].to(first_norm_dev).float()
            pooled_layers[idx_l][good_rows] = (h_good * m[good_rows]).sum(dim=1) / den[good_rows]

    # Rescue overflowed rows in bfloat16 (exact exponent range as float32, NEVER overflows!)
    if n_bad > 0:
        bad_rows = torch.where(bad_mask)[0]
        del hs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        model.to(torch.bfloat16)
        enc_bad = {k: v[bad_rows] for k, v in enc.items()}
        hs_bad = model(**enc_bad, output_hidden_states=True, use_cache=False).hidden_states
        bad_rows_dev = bad_rows.to(first_norm_dev)
        for idx_l, l in enumerate(layers):
            h_b = hs_bad[l].to(first_norm_dev).float()
            h_b = torch.nan_to_num(h_b, nan=0.0, posinf=1e4, neginf=-1e4)
            pooled_layers[idx_l][bad_rows_dev] = (h_b * m[bad_rows_dev]).sum(dim=1) / den[bad_rows_dev]
        del hs_bad
        model.to(dtype)

    out = torch.cat([nm(pooled_layers[idx_l]) for idx_l, nm in enumerate(norms)], dim=-1)
    return out.cpu().numpy(), n_bad

def resolve_head_tensors(heads_dict: dict) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
    """Resolves plain and centered head weights and biases matching American and British naming."""
    W_p, b_p, W_c, b_c = None, None, None, None
    for wk, bk in [
        ("plain.weight", "plain.bias"),
        ("plain_coef", "plain_intercept"),
        ("weight", "bias"),
        ("W", "b"),
    ]:
        if wk in heads_dict and bk in heads_dict:
            W_p, b_p = heads_dict[wk], heads_dict[bk]
            break
    for wk, bk in [
        ("centered.weight", "centered.bias"),
        ("centred.weight", "centred.bias"),
        ("center.weight", "center.bias"),
        ("twin.weight", "twin.bias"),
        ("center_coef", "center_intercept"),
        ("centred_coef", "centred_intercept"),
    ]:
        if wk in heads_dict and bk in heads_dict:
            W_c, b_c = heads_dict[wk], heads_dict[bk]
            break

    if W_p is not None and hasattr(W_p, "cpu"):
        W_p = W_p.cpu().numpy()
    if b_p is not None and hasattr(b_p, "cpu"):
        b_p = b_p.cpu().numpy()
    if W_c is not None and hasattr(W_c, "cpu"):
        W_c = W_c.cpu().numpy()
    if b_c is not None and hasattr(b_c, "cpu"):
        b_c = b_c.cpu().numpy()

    return W_p, b_p, W_c, b_c

def compute_centered_logits(Z_test: np.ndarray, groups: np.ndarray, W_c: np.ndarray, b_c: np.ndarray, src_labels: list) -> Tuple[np.ndarray, np.ndarray]:
    """Computes prompt-topic centered twin logits for valid multi-sample clusters."""
    N = len(Z_test)
    out_lp = np.zeros((N, K), dtype=np.float64)
    if N < 2 or W_c is None:
        return out_lp, np.zeros(N, dtype=bool)
    sz = np.bincount(groups)[groups]
    valid = (sz >= 2) & (sz <= 18)
    if not valid.any():
        return out_lp, valid
    Zc = Z_test.copy()
    for g in np.unique(groups[valid]):
        mask = (groups == g)
        Zc[mask] -= Zc[mask].mean(axis=0, keepdims=True)
    raw_c = Zc[valid] @ W_c.T + b_c
    out_lp[valid] = log_softmax(align_cols(raw_c, src_labels))
    return out_lp, valid

class UniversalLoRATrunkLoader:
    def __init__(self, base_dir: Path, keep_layers: int, layers: List[int], max_len: int = 447):
        if base_dir is not None and not (base_dir / "config.json").exists():
            for sub in base_dir.rglob("config.json"):
                base_dir = sub.parent
                break
        self.base_dir = base_dir
        self.keep_layers = keep_layers
        self.layers = layers
        self.max_len = max_len
        self.total_rescued = 0
        self.tok = AutoTokenizer.from_pretrained(str(base_dir), trust_remote_code=True)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token or "<|endoftext|>"

        cfg = AutoConfig.from_pretrained(str(base_dir), trust_remote_code=True)
        cfg.num_hidden_layers = keep_layers
        if getattr(cfg, "layer_types", None) is not None:
            cfg.layer_types = cfg.layer_types[:keep_layers]
        cfg.use_cache = False

        n_gpus = torch.cuda.device_count()
        if n_gpus >= 2:
            max_memory = {0: "13.5GiB", 1: "13.5GiB"}
            self.model = AutoModel.from_pretrained(
                str(base_dir), config=cfg, device_map="auto", max_memory=max_memory,
                torch_dtype=torch.float16, trust_remote_code=True
            )
        elif n_gpus == 1:
            self.model = AutoModel.from_pretrained(
                str(base_dir), config=cfg, device_map="auto",
                torch_dtype=torch.float16, trust_remote_code=True
            )
        else:
            self.model = AutoModel.from_pretrained(str(base_dir), config=cfg, torch_dtype=torch.float32)

        # Disable final trunk norm so last layer is not double normalized
        if hasattr(self.model, "norm"):
            self.model.norm = nn.Identity()
        elif hasattr(self.model, "model") and hasattr(self.model.model, "norm"):
            self.model.model.norm = nn.Identity()

        self.model.eval()
        self.device_norms = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # nn.LayerNorm with both weight and bias matching heads.safetensors
        hidden_dim = self.model.config.hidden_size
        self.norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim)
            for _ in layers
        ]).to(self.device_norms).float()

        self.norms.eval()
        self.first_dev = next(self.model.parameters()).device

    def snapshot_weights(self, adapter_path: Path) -> dict:
        """Saves CPU fp16 copies of trunk modules modified by adapter."""
        weights_path = adapter_path / "adapter_model.safetensors"
        if not weights_path.exists():
            weights_path = adapter_path / "adapter" / "adapter_model.safetensors"
        if not weights_path.exists():
            return {}
        sd = load_file(str(weights_path))
        state_dict = dict(self.model.named_modules())
        backup = {}
        for k in sd.keys():
            if "lora_A" in k:
                m_name = k.split(".lora_A")[0].replace("base_model.model.", "")
                if m_name in state_dict and hasattr(state_dict[m_name], "weight"):
                    backup[m_name] = state_dict[m_name].weight.detach().cpu().half().clone()
        return backup

    def restore_weights(self, backup: dict):
        """Restores base trunk weights bit-for-bit without FP16 subtraction drift."""
        state_dict = dict(self.model.named_modules())
        with torch.no_grad():
            for m_name, orig in backup.items():
                if m_name in state_dict and hasattr(state_dict[m_name], "weight"):
                    state_dict[m_name].weight.data.copy_(orig.to(state_dict[m_name].weight.device))

    def load_norm_weights(self, heads_sd: dict):
        """Loads LayerNorm weights and biases from head state dict."""
        with torch.no_grad():
            for i, ln in enumerate(self.norms):
                w_k = f'norms.{i}.weight'
                b_k = f'norms.{i}.bias'
                if w_k in heads_sd:
                    ln.weight.copy_(heads_sd[w_k].to(ln.weight.device, ln.weight.dtype))
                if b_k in heads_sd:
                    if hasattr(ln, 'bias') and ln.bias is not None:
                        ln.bias.copy_(heads_sd[b_k].to(ln.bias.device, ln.bias.dtype))

    def _extract_batch_ids(self, id_list: List[List[int]], batch_size: int = 8) -> np.ndarray:
        order = np.argsort([len(x) for x in id_list], kind="stable")
        total_dim = len(self.layers) * self.model.config.hidden_size
        Z = np.zeros((len(id_list), total_dim), dtype=np.float32)

        for s in range(0, len(order), batch_size):
            idx = order[s : s + batch_size]
            enc = self.tok.pad({"input_ids": [id_list[i] for i in idx]}, padding=True, return_tensors="pt")
            enc = {k: v.to(self.first_dev) for k, v in enc.items()}
            out_np, n_bad = forward_and_pool_batch(self.model, enc, self.layers, self.norms, torch.float16, drop_first_token=True)
            self.total_rescued += n_bad
            Z[idx] = out_np
        return Z

    @torch.no_grad()
    def extract_exact(self, texts: List[str], batch_size: int = 8, end_id: int = 151643) -> np.ndarray:
        t0 = time.time()
        self.total_rescued = 0
        raw_ids = self.tok(list(texts), add_special_tokens=False, truncation=True, max_length=self.max_len)["input_ids"]
        ids = [x + [end_id] for x in raw_ids]
        Z = self._extract_batch_ids(ids, batch_size=batch_size)
        print(f"[Extractor] Extracted {len(texts)} samples in {time.time() - t0:.2f}s (Total BF16 Rescued Rows: {self.total_rescued}).")
        return Z

    def free_memory(self):
        del self.model, self.norms
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def extract_deberta_checkpoint(
    ckpt_dir: Path,
    base_deb_dir: Optional[Path],
    texts_test: List[str],
    texts_train: List[str],
    y_train: np.ndarray,
    max_len: int = 512,
    batch_size: int = 8
) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], list]:
    """
    Dedicated feature extractor for DeBERTa-v3-large sequence classification checkpoints.
    Probes drop_first_token and tokenization settings on train.csv to guarantee optimal accuracy.
    """
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    tok_dir = str(base_deb_dir if base_deb_dir else ckpt_dir)
    tok = AutoTokenizer.from_pretrained(tok_dir, use_fast=True)

    head_p = ckpt_dir / "heads.safetensors"
    if not head_p.exists():
        head_p = ckpt_dir / "head.safetensors"
    heads_sd = load_file(str(head_p))
    W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)

    meta_p = ckpt_dir / "meta.json"
    ckpt_labels = LABELS
    layers = [18, 20, 22, 24]
    if meta_p.exists():
        try:
            m = json.load(open(meta_p))
            if "labels" in m:
                ckpt_labels = m["labels"]
            if "layers" in m:
                layers = m["layers"]
        except Exception:
            pass

    model_src = str(ckpt_dir) if (ckpt_dir / "config.json").exists() else (str(base_deb_dir) if base_deb_dir else str(ckpt_dir))
    model = AutoModel.from_pretrained(model_src, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to(dev)
    model.eval()

    hidden_dim = model.config.hidden_size
    norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in layers]).to(dev).float()
    with torch.no_grad():
        for i, ln in enumerate(norms):
            w_k = f"norms.{i}.weight"
            b_k = f"norms.{i}.bias"
            if w_k in heads_sd:
                ln.weight.copy_(heads_sd[w_k].to(dev, ln.weight.dtype))
            if b_k in heads_sd:
                ln.bias.copy_(heads_sd[b_k].to(dev, ln.bias.dtype))
    norms.eval()

    def pool_texts(txt_list: List[str], drop_first: bool = True) -> np.ndarray:
        all_vecs = []
        for s in range(0, len(txt_list), batch_size):
            chunk = txt_list[s : s + batch_size]
            enc = tok(chunk, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
            enc = {k: v.to(dev) for k, v in enc.items()}
            with torch.no_grad():
                hs = model(**enc, output_hidden_states=True).hidden_states
                m = enc["attention_mask"].unsqueeze(-1).float()
                if drop_first:
                    m[:, 0, :] = 0.0
                den = m.sum(dim=1).clamp(min=1.0)
                pooled = [( (hs[l].float() * m).sum(dim=1) / den ) for l in layers]
                out = torch.cat([norms[i](pooled[i]) for i in range(len(layers))], dim=-1)
                all_vecs.append(out.cpu().numpy())
        return np.vstack(all_vecs)

    # Probe drop_first_token on train.csv[:180]
    probe_n = min(180, len(texts_train))
    best_drop = True
    if probe_n > 0 and W_p is not None:
        best_acc = -1.0
        for cand_drop in [True, False]:
            z_pr = pool_texts(texts_train[:probe_n], drop_first=cand_drop)
            preds = align_cols(z_pr @ W_p.T + b_p, ckpt_labels).argmax(axis=1)
            acc = float((preds == y_train[:probe_n]).mean())
            if acc > best_acc:
                best_acc = acc
                best_drop = cand_drop
        print(f"[DeBERTa Extractor] Probed drop_first={best_drop} (accuracy = {best_acc:.4f})")

    Z_test = pool_texts(texts_test, drop_first=best_drop)
    Z_train = pool_texts(texts_train, drop_first=best_drop) if len(texts_train) > 0 else np.zeros((0, len(layers) * hidden_dim), dtype=np.float32)

    del model, norms
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return Z_test, Z_train, W_p, b_p, W_c, b_c, ckpt_labels


In [ ]:
# === MODULE: src/namethatllm/postprocess.py ===
"""
Postprocessing, Fusion & Calibration Engine
===========================================
Pure two-tier convex mixture (85% Main + 15% Complementary),
mass-preserving Claude Specialist Gate, gentle curly quote shield,
and official 300-step exact multiplicative Sinkhorn mass balancing.
"""

from pathlib import Path
from typing import List, Tuple, Optional
import numpy as np
import pandas as pd


def blend_main_and_complementary(
    LOGIT_MAIN: np.ndarray,
    COMP_LIST: List[Tuple[float, np.ndarray]],
    main_ratio: float = 0.85,
    comp_ratio: float = 0.15
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Blends Main Stack and Complementary Stack in probability space:
    P_mix = main_ratio * R_main + comp_ratio * P_comp
    """
    R_main = softmax(LOGIT_MAIN, axis=1)
    if len(COMP_LIST) > 0:
        comp_weight_total = sum(w for w, _ in COMP_LIST)
        P_comp = sum(w * P_c for w, P_c in COMP_LIST) / max(1e-8, comp_weight_total)
        P_mix = main_ratio * R_main + comp_ratio * P_comp
    else:
        P_mix = R_main
    return P_mix, R_main

def apply_claude_specialist_gate(
    P_mix: np.ndarray,
    R_main: np.ndarray,
    claude_spec_logits: Optional[np.ndarray],
    claude_classes: Optional[list] = None,
    labels: list = LABELS,
    claude_cols: list = FAMILIES["claude"],
    gate_weight: float = 0.15,
    weight: Optional[float] = None,
) -> np.ndarray:
    """
    Refines passages predicted in Claude family using specialist head logits.
    Mass-preserving within the Claude family (preserves exact marginal family mass).
    Vectorized NumPy implementation.
    """
    if claude_spec_logits is None:
        return P_mix
    if weight is not None:
        gate_weight = weight

    P_out = P_mix.copy()
    P_claude_head = softmax(claude_spec_logits, axis=-1)
    if claude_classes is not None:
        src_c = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in claude_classes]
        c_idx = [src_c.index(labels[c]) for c in claude_cols]
        P_claude_head = P_claude_head[:, c_idx]

    claude_mask = np.isin(R_main.argmax(axis=1), claude_cols)
    if claude_mask.any():
        rows = np.flatnonzero(claude_mask)
        sub_mix = P_out[np.ix_(rows, claude_cols)]
        tot_mass = sub_mix.sum(axis=1, keepdims=True)
        blended = (1.0 - gate_weight) * sub_mix + gate_weight * (tot_mass * P_claude_head[rows])
        P_out[np.ix_(rows, claude_cols)] = blended
    return P_out

def apply_curly_quote_shield(
    P_mix: np.ndarray,
    test_texts: List[str],
    multiplier: float = 0.36787944117144233  # np.exp(-1.0)
) -> np.ndarray:
    """
    Gentle zero-occurrence guard on Claude and GLM curly quote absence.
    Multiplies target columns by multiplier and re-normalizes rows to simplex.
    """
    claude_and_glm_idx = FAMILIES["claude"] + FAMILIES["glm"]
    for i, txt in enumerate(test_texts):
        txt_s = str(txt)
        if any(q in txt_s for q in ['\u2018', '\u2019', '\u201c', '\u201d']):
            P_mix[i, claude_and_glm_idx] *= multiplier

    # Re-normalize rows to simplex
    P_mix /= P_mix.sum(axis=1, keepdims=True).clip(min=1e-12)
    return P_mix

def sinkhorn_balancing(P: np.ndarray, iters: int = 300) -> np.ndarray:
    """Official 300-step exact multiplicative Sinkhorn column mass balancing."""
    if len(P) < 50:
        return P
    P = np.clip(P.astype(np.float64), 1e-12, 1.0)
    target_col = 1.0 / P.shape[1]
    for _ in range(iters):
        P /= P.sum(axis=1, keepdims=True).clip(min=1e-12)
        P /= (P.sum(axis=0, keepdims=True) / (target_col * P.shape[0])).clip(min=1e-12)
    P /= P.sum(axis=1, keepdims=True).clip(min=1e-12)
    return P

def generate_submission(
    P_final: np.ndarray,
    test_ids: pd.Series,
    working_dir: Path
) -> Tuple[pd.DataFrame, Path, Path]:
    """Generates submission.parquet and submission.csv with exact schema."""
    preds = [LABELS[idx] for idx in P_final.argmax(axis=1)]
    df_sub = pd.DataFrame({"id": test_ids, "model": preds})
    sub_parquet_path = working_dir / "submission.parquet"
    sub_csv_path = working_dir / "submission.csv"

    df_sub.to_parquet(sub_parquet_path, index=False)
    df_sub.to_csv(sub_csv_path, index=False)
    return df_sub, sub_parquet_path, sub_csv_path


In [ ]:
# === MODULE: src/namethatllm/pipeline.py ===
"""
Top-Level Pipeline Orchestrator
===============================
Coordinates the complete end-to-end execution flow:
- Module 1: Dual-Output Lexical Engine (subprocess isolated or in-process)
- Module 2: Neural Trunk Extraction Loop (All 4 trunks and 6 checkpoints)
- Module 3: wPCA DP Posterior (ExactAuthorPosterior) + 384-Dim Spectral Bank
- Module 4: Two-Tier Simplex Blend + Vectorized Claude Gate + Sinkhorn Balancing
"""

import os
import gc
import sys
import json
import time
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from scipy import sparse


def run_submission_pipeline(cfg: Optional[Dict[str, Any]] = None) -> Tuple[pd.DataFrame, Path]:
    """
    Executes the complete production or smoke-test submission pipeline.
    Returns (submission_dataframe, submission_parquet_path).
    """
    if cfg is None:
        cfg = {}

    start_time = time.time()
    is_smoke = os.environ.get("NTL_SMOKE") == "1"

    # 1. Environment & Paths
    if "working_dir" in cfg:
        working_dir = Path(cfg["working_dir"])
    else:
        working_dir = Path("/kaggle/working")
        if not working_dir.exists():
            working_dir = Path("local_data")
    working_dir.mkdir(parents=True, exist_ok=True)

    input_dir = Path(cfg.get("input_dir", "/kaggle/input"))
    if not input_dir.exists():
        input_dir = Path(".")

    if "train_path" in cfg and "test_path" in cfg:
        train_path = Path(cfg["train_path"])
        test_path = Path(cfg["test_path"])
        pool_file = Path(cfg["pool_file"]) if "pool_file" in cfg else None
    else:
        train_path, test_path, pool_file = resolve_dataset_paths(input_dir)

    print(f"[Pipeline] Startup: is_smoke={is_smoke}")
    print(f"[Pipeline] Paths: train={train_path}, test={test_path}, pool={pool_file}")

    df_test = pd.read_csv(test_path)
    test_ids = df_test["id"]
    test_texts = df_test["text"].fillna("").astype(str).tolist()
    N_test = len(test_texts)
    print(f"[Pipeline] Loaded test set: {N_test} rows")

    # 2. MODULE 1: Subprocess Dual-Output Lexical Engine
    run_lex_script = working_dir / "run_lexical.py"
    if run_lex_script.exists() and not is_smoke:
        print("[Pipeline] Module 1: Launching subprocess run_lexical.py...")
        subprocess.run([sys.executable, str(run_lex_script)], check=True)
        log_p_svc_main = np.load(working_dir / "log_p_svc_main.npy")
        p_lex_comp = np.load(working_dir / "p_lex_comp.npy")
        D_tfidf = np.load(working_dir / "D_tfidf.npy")
        groups_085 = np.load(working_dir / "groups_085.npy")
    else:
        print("[Pipeline] Module 1: Executing run_lexical_engine directly...")
        tok_paths = list(input_dir.rglob("vocab.json"))
        qwen_tok_dir = str(tok_paths[0].parent) if tok_paths else ""
        lex_out = run_lexical_engine(train_path, test_path, pool_file, qwen_tok_dir, working_dir)
        log_p_svc_main = lex_out["log_p_svc_main"]
        p_lex_comp = lex_out["p_lex_comp"]
        D_tfidf = lex_out["D_tfidf"]
        groups_085 = lex_out["groups_085"]

    print(f"[Pipeline] Module 1 complete: log_p_svc_main shape = {log_p_svc_main.shape}")

    # Read configuration hierarchies from configs/production_ensemble.json
    main_cfg = cfg.get("main_stack", {})
    comp_cfg = cfg.get("complementary_stack", {})
    fusion_cfg = cfg.get("fusion", {})
    post_cfg = cfg.get("postprocess", {})

    svc_weight = main_cfg.get("linear_svc_weight", 1.00)
    main_members = main_cfg.get("members", {})
    comp_streams = comp_cfg.get("stream_weights", {})
    comp_members = comp_cfg.get("lora_members", {})
    se_floors = comp_cfg.get("se_floors", {
        "qwen3_14b_p2": 0.10, "qwen25_14b_p2": 0.10, "deberta_p2": 0.10,
        "qwen3_14b_p1": 0.35, "instyle": 0.35, "qwen3_8b_claude": 0.35
    })
    prior_scales = comp_cfg.get("prior_scales", PRIOR_SCALES)

    LOGIT_MAIN = log_p_svc_main * svc_weight
    COMP_LIST: List[Tuple[float, np.ndarray]] = [
        (comp_streams.get("lexical", 0.20), p_lex_comp)
    ]
    claude_spec_logits = None
    claude_classes = None

    ckpts = find_checkpoints(input_dir)
    has_neural = len(ckpts) > 0 and not is_smoke

    if has_neural:
        import torch
        from safetensors.torch import load_file

        df_tr = pd.read_csv(train_path)
        u_p = df_tr["prompt_id"].unique()
        sel_prompts = u_p[:min(90, len(u_p))]
        sub_idx = df_tr[df_tr["prompt_id"].isin(sel_prompts)].index.values
        train_sub = df_tr.loc[sub_idx].reset_index(drop=True)
        train_sub_texts = train_sub["text"].fillna("").astype(str).tolist()
        y_sub = np.array([LAB2I[m] for m in train_sub["model"]])
        prompts_sub = train_sub["prompt_id"].values
        N_tr_sub = len(train_sub)
        print(f"[Pipeline] Selected {N_tr_sub} training rows ({len(sel_prompts)} prompts) for wPCA")

        spectral_blocks_tr: List[Tuple[str, np.ndarray]] = []
        spectral_blocks_te: List[Tuple[str, np.ndarray]] = []
        dp_posteriors: List[Tuple[float, np.ndarray]] = []
        comp_lora_posteriors: List[Tuple[float, np.ndarray]] = []

        # ==============================================================================
        # TRUNK 1: Qwen3-14B (40 layers, hidden=5120)
        # ==============================================================================
        base_q3_14b = find_base_model(input_dir, "qwen3", 5120, 40)
        if base_q3_14b and ("qwen3_14b_p2" in ckpts or "qwen3_14b_p1" in ckpts):
            print(f"[Pipeline] Loading Trunk 1: Qwen3-14B from {base_q3_14b}...")
            trunk1 = UniversalLoRATrunkLoader(base_q3_14b, keep_layers=27, layers=[2, 4, 6, 9, 13, 17, 22, 27])

            # Checkpoint 1A: Part 2 (lora_qwen3_14b)
            if "qwen3_14b_p2" in ckpts:
                p2_dir = ckpts["qwen3_14b_p2"]
                backup_p2 = trunk1.snapshot_weights(p2_dir)
                merge_lora(trunk1.model, p2_dir, sign=+1.0)
                heads_sd = load_file(str(p2_dir / "heads.safetensors"))
                trunk1.load_norm_weights(heads_sd)
                W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)

                meta_p = p2_dir / "meta.json"
                p2_labels = LABELS
                if meta_p.exists():
                    try:
                        p2_labels = json.load(open(meta_p)).get("labels", LABELS)
                    except Exception:
                        pass

                Z_all = trunk1.extract_exact(test_texts + train_sub_texts, batch_size=8, end_id=151643)
                Z_te, Z_tr = Z_all[:N_test], Z_all[N_test:]

                # Main Stack contributions
                w_main_p = main_members.get("lora_qwen3_14b", {}).get("plain_weight", 0.40)
                w_main_c = main_members.get("lora_qwen3_14b", {}).get("twin_weight", 0.10)
                lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, p2_labels))
                LOGIT_MAIN += lp_p * w_main_p
                if W_c is not None:
                    lp_c, valid_c = compute_centered_logits(Z_te, groups_085, W_c, b_c, p2_labels)
                    LOGIT_MAIN[valid_c] += lp_c[valid_c] * w_main_c

                # Complementary LoRA stream
                w_comp_p = comp_members.get("lora_qwen3_14b", {}).get("plain_weight", 0.30)
                w_comp_c = comp_members.get("lora_qwen3_14b", {}).get("twin_weight", 0.10)
                comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                if W_c is not None:
                    comp_lora_posteriors.append((w_comp_c, softmax(lp_c, axis=1)))

                # wPCA + DP Posterior + Spectral Bank
                res_p2 = fit_shared_gaussian_lowram(Z_tr, y_sub, prompts_sub, d_target=256, se_floor=se_floors.get("qwen3_14b_p2", 0.10))
                U_te = transform_lowram(Z_te, res_p2)
                lp_dp = scores_exact_dp(U_te, groups_085, res_p2, D_tfidf)
                dp_posteriors.append((0.38, softmax(lp_dp, axis=1)))

                spectral_blocks_tr.append(("qwen3_14b_p2", res_p2["V_top64"]))
                spectral_blocks_te.append(("qwen3_14b_p2", (((Z_te - res_p2["mx"]) / res_p2["sx"]) @ res_p2["V_top64"]).astype(np.float32)))

                trunk1.restore_weights(backup_p2)

            # Checkpoint 1B: Part 1 (lora14b_v1)
            if "qwen3_14b_p1" in ckpts:
                p1_dir = ckpts["qwen3_14b_p1"]
                backup_p1 = trunk1.snapshot_weights(p1_dir)
                merge_lora(trunk1.model, p1_dir, sign=+1.0)
                heads_sd = load_file(str(p1_dir / "heads.safetensors"))
                trunk1.load_norm_weights(heads_sd)
                W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)

                meta_p = p1_dir / "meta.json"
                p1_labels = LABELS
                if meta_p.exists():
                    try:
                        p1_labels = json.load(open(meta_p)).get("labels", LABELS)
                    except Exception:
                        pass

                Z_all = trunk1.extract_exact(test_texts + train_sub_texts, batch_size=8, end_id=151643)
                Z_te, Z_tr = Z_all[:N_test], Z_all[N_test:]

                w_comp_p = comp_members.get("lora14b_v1", {}).get("plain_weight", 0.14)
                w_comp_c = comp_members.get("lora14b_v1", {}).get("twin_weight", 0.05)
                lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, p1_labels))
                comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                if W_c is not None:
                    lp_c, valid_c = compute_centered_logits(Z_te, groups_085, W_c, b_c, p1_labels)
                    comp_lora_posteriors.append((w_comp_c, softmax(lp_c, axis=1)))

                res_p1 = fit_shared_gaussian_lowram(Z_tr, y_sub, prompts_sub, d_target=256, se_floor=se_floors.get("qwen3_14b_p1", 0.35))
                U_te = transform_lowram(Z_te, res_p1)
                lp_dp = scores_exact_dp(U_te, groups_085, res_p1, D_tfidf)
                dp_posteriors.append((0.08, softmax(lp_dp, axis=1)))

                spectral_blocks_tr.append(("qwen3_14b_p1", res_p1["V_top64"]))
                spectral_blocks_te.append(("qwen3_14b_p1", (((Z_te - res_p1["mx"]) / res_p1["sx"]) @ res_p1["V_top64"]).astype(np.float32)))

                trunk1.restore_weights(backup_p1)

            trunk1.free_memory()

        # ==============================================================================
        # TRUNK 2: Qwen2.5-14B (48 layers, hidden=5120)
        # ==============================================================================
        base_q25_14b = find_base_model(input_dir, "qwen2", 5120, 48)
        if base_q25_14b and "qwen25_14b_p2" in ckpts:
            print(f"[Pipeline] Loading Trunk 2: Qwen2.5-14B from {base_q25_14b}...")
            trunk2 = UniversalLoRATrunkLoader(base_q25_14b, keep_layers=32, layers=[3, 6, 10, 14, 18, 23, 27, 32])
            p2_dir = ckpts["qwen25_14b_p2"]
            merge_lora(trunk2.model, p2_dir, sign=+1.0)
            heads_sd = load_file(str(p2_dir / "heads.safetensors"))
            trunk2.load_norm_weights(heads_sd)
            W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)

            meta_p = p2_dir / "meta.json"
            p2_labels = LABELS
            if meta_p.exists():
                try:
                    p2_labels = json.load(open(meta_p)).get("labels", LABELS)
                except Exception:
                    pass

            Z_all = trunk2.extract_exact(test_texts + train_sub_texts, batch_size=8, end_id=151643)
            Z_te, Z_tr = Z_all[:N_test], Z_all[N_test:]

            w_main_p = main_members.get("lora_qwen25_14b", {}).get("plain_weight", 0.20)
            w_main_c = main_members.get("lora_qwen25_14b", {}).get("twin_weight", 0.10)
            lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, p2_labels))
            LOGIT_MAIN += lp_p * w_main_p
            if W_c is not None:
                lp_c, valid_c = compute_centered_logits(Z_te, groups_085, W_c, b_c, p2_labels)
                LOGIT_MAIN[valid_c] += lp_c[valid_c] * w_main_c

            w_comp_p = comp_members.get("lora_qwen25_14b", {}).get("plain_weight", 0.20)
            w_comp_c = comp_members.get("lora_qwen25_14b", {}).get("twin_weight", 0.08)
            comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
            if W_c is not None:
                comp_lora_posteriors.append((w_comp_c, softmax(lp_c, axis=1)))

            res_q25 = fit_shared_gaussian_lowram(Z_tr, y_sub, prompts_sub, d_target=256, se_floor=se_floors.get("qwen25_14b_p2", 0.10))
            U_te = transform_lowram(Z_te, res_q25)
            lp_dp = scores_exact_dp(U_te, groups_085, res_q25, D_tfidf)
            dp_posteriors.append((0.24, softmax(lp_dp, axis=1)))

            spectral_blocks_tr.append(("qwen25_14b_p2", res_q25["V_top64"]))
            spectral_blocks_te.append(("qwen25_14b_p2", (((Z_te - res_q25["mx"]) / res_q25["sx"]) @ res_q25["V_top64"]).astype(np.float32)))

            trunk2.free_memory()

        # ==============================================================================
        # TRUNK 3: Qwen3-8B (36 layers, hidden=4096)
        # ==============================================================================
        base_q3_8b = find_base_model(input_dir, "qwen3", 4096, 36)
        if base_q3_8b and ("instyle" in ckpts or "qwen3_8b_claude" in ckpts):
            print(f"[Pipeline] Loading Trunk 3: Qwen3-8B from {base_q3_8b}...")
            trunk3 = UniversalLoRATrunkLoader(base_q3_8b, keep_layers=24, layers=[2, 5, 8, 11, 14, 18, 21, 24])

            # In-Style LoRA
            if "instyle" in ckpts:
                instyle_dir = ckpts["instyle"]
                backup_ins = trunk3.snapshot_weights(instyle_dir)
                merge_lora(trunk3.model, instyle_dir, sign=+1.0)
                heads_sd = load_file(str(instyle_dir / "heads.safetensors")) if (instyle_dir / "heads.safetensors").exists() else {}
                trunk3.load_norm_weights(heads_sd)
                W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)

                Z_all = trunk3.extract_exact(test_texts + train_sub_texts, batch_size=8, end_id=151643)
                Z_te, Z_tr = Z_all[:N_test], Z_all[N_test:]

                if W_p is not None:
                    w_comp_p = comp_members.get("instyle", {}).get("plain_weight", 0.22)
                    lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, LABELS))
                    comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))

                res_ins = fit_shared_gaussian_lowram(Z_tr, y_sub, prompts_sub, d_target=256, se_floor=se_floors.get("instyle", 0.35))
                U_te = transform_lowram(Z_te, res_ins)
                lp_dp = scores_exact_dp(U_te, groups_085, res_ins, D_tfidf)
                dp_posteriors.append((0.10, softmax(lp_dp, axis=1)))

                spectral_blocks_tr.append(("instyle", res_ins["V_top64"]))
                spectral_blocks_te.append(("instyle", (((Z_te - res_ins["mx"]) / res_ins["sx"]) @ res_ins["V_top64"]).astype(np.float32)))

                trunk3.restore_weights(backup_ins)

            # Claude Head LoRA
            if "qwen3_8b_claude" in ckpts:
                claude_dir = ckpts["qwen3_8b_claude"]
                backup_cl = trunk3.snapshot_weights(claude_dir)
                merge_lora(trunk3.model, claude_dir, sign=+1.0)
                head_file = claude_dir / "claude_head.safetensors"
                if not head_file.exists():
                    head_file = claude_dir / "heads.safetensors"
                heads_sd = load_file(str(head_file)) if head_file.exists() else {}
                trunk3.load_norm_weights(heads_sd)
                W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)

                meta_p = claude_dir / "meta.json"
                if meta_p.exists():
                    try:
                        claude_classes = json.load(open(meta_p)).get("classes", None)
                    except Exception:
                        pass

                Z_all = trunk3.extract_exact(test_texts + train_sub_texts, batch_size=8, end_id=151643)
                Z_te, Z_tr = Z_all[:N_test], Z_all[N_test:]

                if W_p is not None:
                    claude_spec_logits = Z_te @ W_p.T + b_p
                    w_comp_p = comp_members.get("ntl_claude_head", {}).get("plain_weight", 0.14)
                    lp_p = log_softmax(align_cols(claude_spec_logits, claude_classes or LABELS))
                    comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))

                trunk3.restore_weights(backup_cl)

            trunk3.free_memory()

        # ==============================================================================
        # TRUNK 4: DeBERTa-v3-Large
        # ==============================================================================
        if "deberta_p2" in ckpts:
            deb_dir = ckpts["deberta_p2"]
            base_deb_dir = find_base_model(input_dir, "deberta", 1024, 24)
            print(f"[Pipeline] Loading Trunk 4: DeBERTa-v3 from {deb_dir}...")
            Z_te_deb, Z_tr_deb, W_p_deb, b_p_deb, W_c_deb, b_c_deb, deb_labels = extract_deberta_checkpoint(
                deb_dir, base_deb_dir, test_texts, train_sub_texts, y_sub
            )
            if W_p_deb is not None:
                w_main_deb = main_members.get("deberta_v3_large", {}).get("plain_weight", 0.25)
                lp_deb = log_softmax(align_cols(Z_te_deb @ W_p_deb.T + b_p_deb, deb_labels))
                LOGIT_MAIN += lp_deb * w_main_deb

                w_comp_deb = comp_members.get("deberta_v3_large", {}).get("plain_weight", 0.18)
                comp_lora_posteriors.append((w_comp_deb, softmax(lp_deb, axis=1)))

            res_deb = fit_shared_gaussian_lowram(Z_tr_deb, y_sub, prompts_sub, d_target=256, se_floor=se_floors.get("deberta_p2", 0.10))
            U_te = transform_lowram(Z_te_deb, res_deb)
            lp_dp = scores_exact_dp(U_te, groups_085, res_deb, D_tfidf)
            dp_posteriors.append((0.20, softmax(lp_dp, axis=1)))

            spectral_blocks_tr.append(("deberta_p2", res_deb["V_top64"]))
            spectral_blocks_te.append(("deberta_p2", (((Z_te_deb - res_deb["mx"]) / res_deb["sx"]) @ res_deb["V_top64"]).astype(np.float32)))

        # Add Secondary LoRAs to COMP_LIST
        if comp_lora_posteriors:
            tot_w = sum(w for w, _ in comp_lora_posteriors)
            P_lora_comb = sum(w * P for w, P in comp_lora_posteriors) / max(1e-8, tot_w)
            COMP_LIST.append((comp_streams.get("lora_and_twins", 0.35), P_lora_comb))

        # Add DP Posteriors to COMP_LIST
        if dp_posteriors:
            tot_dp_w = sum(w for w, _ in dp_posteriors)
            P_dp_comb = sum(w * P_dp for w, P_dp in dp_posteriors) / max(1e-8, tot_dp_w)
            COMP_LIST.append((comp_streams.get("dp_posteriors", 0.30), P_dp_comb))

        # Fit Joint Spectral Bank
        if spectral_blocks_tr and spectral_blocks_te:
            P_spectral, oof_score = fit_joint_spectral_bank(
                spectral_blocks_tr, spectral_blocks_te, y_sub, prompts_sub,
                prior_scales=prior_scales, n_splits=5
            )
            print(f"[Pipeline] Joint Spectral Bank fitted (OOF BAcc = {oof_score:.4f})")
            COMP_LIST.append((comp_streams.get("spectral_bank", 0.15), P_spectral))

    elif is_smoke:
        print("[Pipeline] Smoke mode active: synthesizing high-fidelity mock representations...")
        np.random.seed(42)
        mock_p_lora = softmax(np.random.randn(N_test, K), axis=1)
        mock_p_dp = softmax(np.random.randn(N_test, K), axis=1)
        mock_p_spec = softmax(np.random.randn(N_test, K), axis=1)
        COMP_LIST.append((comp_streams.get("lora_and_twins", 0.35), mock_p_lora))
        COMP_LIST.append((comp_streams.get("dp_posteriors", 0.30), mock_p_dp))
        COMP_LIST.append((comp_streams.get("spectral_bank", 0.15), mock_p_spec))

    # 4. MODULE 4: Convex Blend, Claude Specialist Gate, Curly Quote Guard & Sinkhorn
    print("[Pipeline] Module 4: Fusing Main and Complementary stacks...")
    main_ratio = fusion_cfg.get("main_ratio", 0.85)
    comp_ratio = fusion_cfg.get("comp_ratio", 0.15)
    P_mix, R_main = blend_main_and_complementary(LOGIT_MAIN, COMP_LIST, main_ratio=main_ratio, comp_ratio=comp_ratio)

    # Claude Specialist Gate (Vectorized)
    gate_weight = post_cfg.get("claude_specialist_weight", 0.15)
    P_mix = apply_claude_specialist_gate(
        P_mix=P_mix,
        R_main=R_main,
        claude_spec_logits=claude_spec_logits,
        claude_classes=claude_classes,
        labels=LABELS,
        claude_cols=FAMILIES["claude"],
        gate_weight=gate_weight
    )

    # Curly Quote Shield (gentle -1.0 penalty in log-space = exp(-1.0) probability multiplier)
    default_curly_multiplier = float(np.exp(-1.0))
    multiplier = post_cfg.get("curly_quote_multiplier", default_curly_multiplier)
    P_mix = apply_curly_quote_shield(P_mix, test_texts, multiplier=multiplier)

    # 300-Step Sinkhorn Mass Balancing
    sinkhorn_iters = post_cfg.get("sinkhorn_iterations", 300)
    P_final = sinkhorn_balancing(P_mix, iters=sinkhorn_iters)

    # Generate Submission Artifacts
    df_sub, out_parquet, out_csv = generate_submission(P_final, test_ids, working_dir)
    elapsed = time.time() - start_time
    print(f"[Pipeline] Submission generated successfully in {elapsed:.2f}s: {out_parquet} ({len(df_sub)} rows)")

    return df_sub, out_parquet


In [ ]:
if __name__ == '__main__':
    run_submission_pipeline(ENSEMBLE_CONFIG)
